# Tutorial 0 — Start here

Estimated time: 10 minutes. No simulation, no model runs — this is orientation.

This repository reproduces and extends **Neve-Oz, Sherman & Raveh, "Bayesian
metamodeling of early T-cell antigen receptor signaling accounts for its nanoscale
activation patterns"** (*Frontiers in Immunology*, 2024).

There are two ways to read it, and they answer different questions. This notebook helps
you pick, and checks that the one you pick will actually run on your machine.

> **Citation.** The paper's DOI is `10.3389/fimmu.2024.1412221`
> ([resolves to Frontiers](https://doi.org/10.3389/fimmu.2024.1412221)). An earlier
> `10.3389/fimmu.2024.1437672` appeared in this repo's README and was wrong — it 404s.
> Checked against the DOI resolver and PubMed (PMID 39524449) rather than assumed.


## The two tracks

**Track A — the model track.** How does one biophysical model work? Currently this
means kinetic segregation: the biology, the energy function, the Monte Carlo scheme,
what each parameter does, and how to measure the result without fooling yourself. Runs
on numpy + a C binary. **No framework needed.**

**Track B — the metamodel track.** How do four separately-built models get combined
into one joint posterior? Sweeps, surrogates, couplings, and the paper's figures.
Drives the `bayesmm` CLI. **Needs the `bayesian-metamodeling` framework**, and PyMC for
the surrogate and inference steps.

| | Track A — model | Track B — metamodel |
|---|---|---|
| Location | `models/kinetic_segregation/KS_1` … `KS_5` | `01_explore_models` … `04_reproduce_figures` |
| Question | *why does this model behave like this?* | *how do models constrain each other?* |
| Needs | numpy, matplotlib, a C compiler | `bayesmm`; PyMC for `02`/`03` |
| Runtime | seconds per notebook | minutes (sampling) |
| Executed in CI | yes | no (framework lives in the parent repo) |

## Which should you read?

| If you want to… | Go to |
|---|---|
| Understand what kinetic segregation *is* | **KS 1** |
| See the energy terms and the MC algorithm | **KS 2** |
| Run the simulator and read its output | **KS 3** |
| Know what κ, CD45 height, or binding mode do | **KS 4** |
| Choose a depletion metric, or avoid the CLI traps | **KS 5** |
| Run all four partial models once | `01_explore_models` |
| Fit surrogates to sweep data | `02_fit_surrogates` |
| Sample the coupled joint posterior | `03_metamodel_inference` |
| Reproduce the pTCR ring figure | `04_reproduce_figures` |

**If you are new, read KS 1 first even if you came for the metamodel.** Track B treats
each partial model as a box that emits a number; Track A is where you learn what that
number means and how badly it can mislead you.

## What the four partial models are

| # | Model | Question | Contributes |
|---|---|---|---|
| 1 | Membrane topography | Where are the membranes close enough to count as contact? | contact geometry |
| 2 | **Kinetic segregation** | Where does CD45 end up, given that geometry? | `depletion_width_nm` |
| 3 | Lck activity | How far from the CD45 boundary does active kinase persist? | `mean_lck_activity` |
| 4 | TCR phosphorylation | Where do phosphorylated ITAMs accumulate? | pTCR profile |

The scientific payoff is a **ring**: CD45 is most depleted at the centre of the contact,
but active Lck decays over ~70 nm from the boundary, so the product of "kinase present"
and "phosphatase absent" peaks in an annulus rather than at the centre.

They are **declared as coupled, not chained** — the spec states which variables are the
same physical quantity, rather than piping each output forward as a fixed input.
*How much the tooling makes of that declaration depends on which sampler you ask for*:
the default performs forward propagation through the couplings, while `--method joint`
conditions on the surrogate likelihoods as well, so a coupling informs both of its
variables. `03_metamodel_inference` spells out the difference. Evidence about one sharpens the others.

## Track B: run them in order

01 → 02 → 03 → 04, and the middle arrow is load-bearing:

| notebook | needs | produces | time |
|---|---|---|---|
| `01_explore_models` | the built `ks_gpu` binary | one run of each of the four models | ~1 min |
| `02_fit_surrogates` | 01's models working | sweeps, then **four published surrogates** in `artifacts/` | ~2 min |
| `03_metamodel_inference` | **02's published artifacts** | the coupled IR and its draws | ~10 s |
| `04_reproduce_figures` | nothing from the others | the pTCR ring figure | ~10 s |

**`03` cannot run until `02` has.** The metamodel spec references four surrogate
artifacts by name, and 02's last step is what publishes them. Running 03 on a fresh
checkout fails with `Could not resolve surrogate_ref` — that is the dependency talking,
not a broken notebook.

`04` is independent: it computes the ring analytically, so you can read it at any point.

If a notebook fails, read its **self-check** cell at the bottom. Each one asserts the
scientific artifact it was supposed to produce — not merely that the code ran — so the
assertion message usually names the actual problem.

## Will it run here?

The check below is honest about which track is available to you. Neither answer is
wrong — Track A deliberately avoids the framework so it works in a bare checkout.

**Windows, macOS and Linux are all supported**, and all three are built and tested in CI
on every push. Track A compiles a small C program the first time you run it, so it needs
CMake and a C/C++ compiler:

- **Windows** — Visual Studio 2022 **Build Tools** with the *Desktop development with
  C++* workload. This is the piece people miss: having Visual Studio installed is not
  enough if that workload was never ticked. The check below detects the workload
  specifically (not just "is a compiler on `PATH`", which is always false on Windows even
  when everything is fine) and prints the exact `winget` command if it is absent.
- **macOS** — `xcode-select --install`, plus CMake.
- **Linux** — `build-essential` and CMake.

If anything is missing, the cell prints the install command for *your* platform. Full
details: [build prerequisites](../README.md#build-prerequisites-windows-macos-linux).

In [1]:
import shutil
import subprocess
import sys
from pathlib import Path


def find_repo_root(start=None):
    here = (start or Path.cwd()).resolve()
    for cand in (here, *here.parents):
        if (cand / "models" / "kinetic_segregation" / "CMakeLists.txt").is_file():
            return cand
    raise RuntimeError(f"could not locate the tcr_signaling repo above {here}")


ROOT = find_repo_root()
KS_DIR = ROOT / "models" / "kinetic_segregation"

# The model's own resolver. It knows what the compiled artifacts are called on
# this OS (ks_gpu vs ks_gpu.exe) and how to detect a compiler CMake can use --
# which on Windows is NOT "is c++ on PATH": MSVC lives outside PATH and is found
# through the registry, so the naive check calls a working install broken.
sys.path.insert(0, str(KS_DIR))
import ks_build  # noqa: E402


def have_module(name):
    try:
        __import__(name)
        return True
    except Exception:
        return False


def have_cli(name):
    """Is this console script runnable? `which` first -- subprocess raises if absent."""
    if shutil.which(name) is None:
        return False
    try:
        return subprocess.run([name, "--version"], capture_output=True, text=True).returncode == 0
    except OSError:
        return False


binary = ks_build.find_binary() is not None
cmake, compiler = ks_build.have_cmake(), ks_build.have_compiler()

checks = [
    ("python", True, sys.version.split()[0]),
    ("numpy", have_module("numpy"), "Track A"),
    ("matplotlib", have_module("matplotlib"), "Track A"),
    ("cmake", cmake, "Track A (configures the build)"),
    ("C/C++ compiler", compiler, "Track A (compiles the model)"),
    ("model binary built", binary, f"Track A ({ks_build.binary_name()}; built on first use)"),
    ("bayesmm", have_cli("bayesmm"), "Track B"),
    ("pymc", have_module("pymc"), "Track B: notebooks 02, 03"),
    ("sbi", have_module("sbi"), "Track B: optional sbi_npe backend"),
]
width = max(len(n) for n, _, _ in checks)
for name, ok, note in checks:
    print(f"  {'OK ' if ok else '-- '} {name:<{width}}   {note}")

track_a = all([have_module("numpy"), have_module("matplotlib"), (cmake and compiler) or binary])
track_b = have_cli("bayesmm")
print()
print(f"Track A (kinetic segregation) : {'ready' if track_a else 'NOT ready'}")
print(f"Track B (metamodel)           : {'ready' if track_b else 'NOT ready'}")

if not track_a:
    # Platform-correct instructions -- the winget line on Windows, xcode-select
    # on macOS, apt/conda on Linux.
    print("\nTo make Track A ready:\n")
    print(ks_build.toolchain_hint())
if not track_b:
    print("\nTo make Track B ready: install the framework (`pip install -e .` in the")
    print("parent metamodeling repo), then `pip install pymc` for notebooks 02 and 03.")
if track_a and not track_b:
    print("\n-> Start at models/kinetic_segregation/KS_1_Kinetic_Segregation.ipynb")

  OK  python               3.14.6
  OK  numpy                Track A
  OK  matplotlib           Track A
  OK  cmake                Track A (configures the build)
  OK  C/C++ compiler       Track A (compiles the model)
  OK  model binary built   Track A (ks_gpu; built on first use)
  OK  bayesmm              Track B
  --  pymc                 Track B: notebooks 02, 03
  --  sbi                  Track B: optional sbi_npe backend



Track A (kinetic segregation) : ready
Track B (metamodel)           : ready


## Vocabulary

- **Partial model** — one of the four biophysical models, independently parameterised
  and independently validated.
- **Sweep / DOE** — running a model over a planned set of input combinations.
- **Surrogate** — a fast *probabilistic* stand-in fit to a sweep. It predicts the
  model's output at unseen inputs *and reports its own uncertainty*; that second part is
  what makes it usable inside a Bayesian metamodel.
- **Coupling** — an assertion that two variables in different models are the same
  physical quantity, or related by a known transform. `gaussian_link` says "these should
  agree, to within σ"; `deterministic` says "these are equal by construction".
- **Joint posterior** — the distribution over all models' variables *after* the
  couplings have been imposed, narrower than any model alone. That is the goal.
  **Caveat, so the word does not mislead you:** `bayesmm meta sample` has two modes.
  The default, `--method propagate`, draws from the priors and applies the couplings
  as a post-draw transform — forward uncertainty propagation, not conditioning.
  `--method joint` runs a Metropolis chain over the full joint density *including the
  fitted surrogates*, which is the one that earns the word. `03_metamodel_inference`
  shows both and how to tell them apart in the numbers.
- **Depletion width** — the KS model's headline observable: the radial gap between the
  CD45 and TCR distributions. KS 3 and KS 5 explain when it means what you think.

## Three things worth knowing before you start

1. **Grid resolution is not a speed/quality dial in the KS model.** If
   `dx = patch_size/grid_size` is much larger than `sigma_r` (2 nm), the pMHC influence
   weight becomes exactly zero and the TCR attraction switches off entirely. KS 3
   measures this. The checked-in specs currently sit in that regime.
2. **A metric reporting "no effect" may be saturated rather than insensitive.** KS 4
   shows `u_assoc` looking inert on one observable and clearly active on another.
3. **Two of the eight depletion metrics are `null` unless you pass
   `--monitor-binding`** — and they happen to be the two that survive an off-centre
   contact. KS 5 covers this.

In [2]:
assert ROOT.is_dir() and KS_DIR.is_dir()
assert (ROOT / "notebooks" / "models" / "kinetic_segregation").is_dir(), \
    "Track A notebooks are missing"
assert len(list((ROOT / "notebooks" / "models" / "kinetic_segregation").glob("KS_*.ipynb"))) >= 5
print("[Tutorial_0 self-check OK]")

[Tutorial_0 self-check OK]
